# Vignette C: Perturbation-response prediction with LEMBAS

This vignette shows the full perturbation-response workflow using the macrophage dataset from
[Nilsson et al. 2022](https://doi.org/10.1038/s41467-022-30684-y).
We will:

1. Download the prior knowledge network and matched ligand / TF activity data
2. Format and construct the network graph
3. Split conditions into train and test sets
4. Fit a mean-response baseline and a ridge baseline
5. Fit the LEMBAS-RNN model (requires PyTorch)
6. Evaluate all three models per-readout and per-condition

> **Note:** step 5 requires `torch`. Install it with `pip install torch` if needed.

In [ ]:
import pandas as pd
import networkcommons as nc
from networkcommons.utils import lembas_format_network, network_from_df
from networkcommons.methods import (
    split_perturbation_data,
    run_mean_response_baseline,
    run_ridge_baseline,
    run_lembas_rnn,
    evaluate_predictions,
)

## 1. Load LEMBAS macrophage data

The macrophage dataset contains 23 stimulation conditions, 13 ligands and
89 TF readouts. The prior knowledge network has ~5 700 signed edges.
All files are cached locally after the first download.

In [ ]:
net_df  = nc.data.omics.lembas_network('macrophage')
ligands = nc.data.omics.lembas_ligands('macrophage')
tfs     = nc.data.omics.lembas_tfs('macrophage')

print('Network edges :', len(net_df))
print('Conditions    :', len(ligands))
print('Ligand inputs :', ligands.shape[1])
print('TF readouts   :', tfs.shape[1])

In [ ]:
net_df.head()

In [ ]:
ligands.head()

## 2. Build the prior knowledge network

`lembas_format_network` converts the binary `stimulation` / `inhibition` columns
into a single signed `mode_of_action` column (+1 / −1 / 0.1 for unknown).
`network_from_df` then wraps the edge table in a NetworkX `DiGraph`.

In [ ]:
net_df = lembas_format_network(net_df)
graph  = network_from_df(net_df, source_col='source', target_col='target')

print('Nodes:', graph.number_of_nodes())
print('Edges:', graph.number_of_edges())

## 3. Train / test split

`split_perturbation_data` aligns the ligand and TF matrices on shared
condition indices before splitting, so row order mismatches are handled
automatically.

In [ ]:
x_train, x_test, y_train, y_test = split_perturbation_data(
    ligands,
    tfs,
    test_size=0.2,
    seed=42,
)

print('Train conditions:', len(x_train))
print('Test  conditions:', len(x_test))

## 4. Baseline models

### 4a. Mean-response baseline

Predicts every test condition as the column-wise training mean — a
useful lower bound that ignores which ligand was applied.

In [ ]:
mean_result = run_mean_response_baseline(y_train, x_test)
mean_metrics = evaluate_predictions(y_test, mean_result['predictions'])
mean_metrics

### 4b. Ridge regression baseline

Fits a linear map from ligand inputs to TF outputs using L2 regularisation.
This captures linear dose–response relationships without any network structure.

In [ ]:
ridge_result = run_ridge_baseline(x_train, y_train, x_test)
ridge_metrics = evaluate_predictions(y_test, ridge_result['predictions'])
ridge_metrics

## 5. LEMBAS-RNN

The LEMBAS recurrent neural network constrains its weight matrix to the
signed prior knowledge network. Node states are propagated with the
Michaelis-Menten-like (MML) activation until steady state, and the
training loss combines MSE, sign regularisation and a uniform-distribution
penalty that keeps node states spread across [0, 1].

> **Requires `torch`.**

In [ ]:
lembas_result = run_lembas_rnn(
    graph,
    x_train,
    y_train,
    x_test,
    epochs=100,
    n_steps=50,
    learning_rate=1e-3,
    seed=42,
)

import matplotlib.pyplot as plt
plt.plot(lembas_result['loss_history'])
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('LEMBAS training curve')
plt.tight_layout()
plt.show()

## 6. Evaluation

### 6a. Per-readout metrics

`axis='readout'` (default) reports MSE, MAE and Pearson correlation for
each TF across all test conditions, plus an `__all__` aggregate row.

In [ ]:
lembas_metrics_readout = evaluate_predictions(
    y_test,
    lembas_result['predictions'],
    axis='readout',
)
lembas_metrics_readout.sort_values('pearson', ascending=False)

### 6b. Per-condition metrics

`axis='condition'` reports the same metrics for each held-out condition
across all TF readouts — useful for spotting which stimulations the model
finds hardest to predict.

In [ ]:
lembas_metrics_condition = evaluate_predictions(
    y_test,
    lembas_result['predictions'],
    axis='condition',
)
lembas_metrics_condition.sort_values('mse', ascending=False)

### 6c. Model comparison

Comparing `__all__` Pearson correlation across the three models.

In [ ]:
comparison = pd.DataFrame({
    'Mean baseline' : mean_metrics.loc['__all__'],
    'Ridge'         : ridge_metrics.loc['__all__'],
    'LEMBAS-RNN'    : lembas_metrics_readout.loc['__all__'],
}).T

comparison